# E2 — Ranking Accuracy & E3 — Winner Plan Quality

Notebook đánh giá Cost Model dựa trên dữ liệu filtered CSV.

**Cấu trúc:**
- Load `gpt2_results_filtered.csv` và `qwen25_results_filtered.csv`
- E2: Spearman ρ, Kendall τ, Top-1/Top-k accuracy, MAPE (theo từng world_size)
- E3: Oracle plan, Regret, Winner accuracy
- Visualization: scatter plots, rank comparison bars

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

RESULTS_DIR = "/home/ductm27/ColossalAI/examples/language/gpt/experiments/auto_parallel/results"

# Load filtered CSVs
gpt2 = pd.read_csv(os.path.join(RESULTS_DIR, "gpt2_results_filtered.csv"))
qwen = pd.read_csv(os.path.join(RESULTS_DIR, "qwen25_results_filtered.csv"))

print("GPT-2 rows:", len(gpt2))
print("Qwen rows:", len(qwen))

# Quick peek
display(gpt2.head(3))

GPT-2 rows: 29
Qwen rows: 23


,filename,model,world_size,pp,tp,dp,dp_outside,layers,hidden,heads,...,compute_ms,bubble_ms,tp_comm_ms,pp_comm_ms,dp_comm_ms,execution_overhead_ms,embedding_ms,lm_head_ms,actual_avg_ms,ratio_actual_estimate
0,gpt2_2gpu_24L_1024H_16B_pp1_tp1_dp2_no_dp_outs...,gpt2,2,1,1,2,False,24,1024,16,...,381.259781,0.000000,0.0,0.000000,64.333853,67.247922,0.332054,2.988486,515.949188,1.000
1,gpt2_2gpu_24L_1024H_16B_pp2_tp1_dp1_no_dp_outs...,gpt2,2,2,1,1,False,24,1024,16,...,340.933640,37.881516,0.0,1.403381,0.000000,23.173961,5.488051,49.392461,357.243853,0.780
2,gpt2_4gpu_24L_1024H_16B_pp1_tp1_dp4.json,gpt2,4,1,1,4,True,24,1024,16,...,381.923332,0.000000,0.0,0.000000,464.606714,67.247922,0.328322,2.954894,906.753732,0.989


## Helper functions

In [ ]:
def rank_by(series, ascending=True):
    """Return rank (1=best). Smaller time = better rank."""
    return series.rank(ascending=ascending).astype(int)

def evaluate_group(df):
    """
    Compute ranking metrics for a single group (same model, world_size, ...).
    Returns dict of metrics.
    """
    n = len(df)
    if n < 2:
        return None
    
    est = df["estimated_total_ms"].values
    act = df["actual_avg_ms"].values
    
    # Ranks (1 = best = smallest time)
    est_rank = rank_by(df["estimated_total_ms"])
    act_rank = rank_by(df["actual_avg_ms"])
    
    # Spearman
    rho, pval = stats.spearmanr(est_rank, act_rank)
    
    # Kendall
    tau, _ = stats.kendalltau(est_rank, act_rank)
    
    # Top-1 accuracy
    best_est = est_rank.idxmin()
    best_act = act_rank.idxmin()
    top1 = 1 if best_est == best_act else 0
    
    # Top-k accuracy (k=min(3, n-1))
    k = min(3, n - 1) if n > 1 else 0
    topk_est = set(est_rank.nsmallest(k).index)
    topk_act = set(act_rank.nsmallest(k).index)
    topk = len(topk_est & topk_act) / k if k > 0 else np.nan
    
    # MAPE
    mape = np.mean(np.abs((est - act) / act)) * 100
    
    # Winner plan (E3)
    oracle_idx = act_rank.idxmin()
    winner_idx = est_rank.idxmin()
    oracle_time = df.loc[oracle_idx, "actual_avg_ms"]
    winner_time = df.loc[winner_idx, "actual_avg_ms"]
    regret = (winner_time - oracle_time) / oracle_time * 100 if oracle_time > 0 else np.nan
    
    return {
        "n_candidates": n,
        "spearman_rho": round(rho, 3) if not np.isnan(rho) else None,
        "spearman_pval": round(pval, 4) if pval is not None else None,
        "kendall_tau": round(tau, 3) if not np.isnan(tau) else None,
        "top1_acc": top1,
        "topk_acc": round(topk, 3) if not np.isnan(topk) else None,
        "mape_pct": round(mape, 1),
        "oracle_plan": df.loc[oracle_idx, ["pp", "tp", "dp"]].to_dict(),
        "winner_plan": df.loc[winner_idx, ["pp", "tp", "dp"]].to_dict(),
        "oracle_actual_ms": round(oracle_time, 1),
        "winner_actual_ms": round(winner_time, 1),
        "regret_pct": round(regret, 1) if not np.isnan(regret) else None,
        "winner_correct": top1,
    }

## E2 — Ranking Evaluation (per world_size)

In [ ]:
def evaluate_model(df, model_name):
    """Run E2+E3 for all world_size groups in a model dataframe."""
    groups = df.groupby(["world_size"])
    records = []
    for ws, gdf in groups:
        metrics = evaluate_group(gdf)
        if metrics:
            metrics["model"] = model_name
            metrics["world_size"] = ws
            records.append(metrics)
    return pd.DataFrame(records)

e2_gpt2 = evaluate_model(gpt2, "gpt2")
e2_qwen = evaluate_model(qwen, "qwen25")
e2_all = pd.concat([e2_gpt2, e2_qwen], ignore_index=True)

# Pretty display
print('\n=== E2a — Per world_size ===')
display_cols = ['model', 'world_size', 'n_candidates', 'spearman_rho', 'kendall_tau', 'top1_acc', 'topk_acc', 'mape_pct']
display(e2_all[display_cols])

# E2b — Combined across all world_sizes for each model
def evaluate_combined(df, model_name):
    metrics = evaluate_group(df)
    if metrics:
        metrics['model'] = model_name
        metrics['world_size'] = 'all'
    return metrics

combined_records = []
for df_model, name in [(gpt2, 'gpt2'), (qwen, 'qwen25')]:
    m = evaluate_combined(df_model, name)
    if m:
        combined_records.append(m)
e2_combined = pd.DataFrame(combined_records)

print('\n=== E2b — Combined across all world_sizes ===')
display(e2_combined[display_cols])

## E3 — Winner Plan Quality

In [ ]:
e3_cols = ['model', 'world_size', 'winner_plan', 'oracle_plan', 'winner_correct', 'oracle_actual_ms', 'winner_actual_ms', 'regret_pct']
print('\n=== E3a — Per world_size ===')
display(e2_all[e3_cols])
print('\n=== E3b — Combined across all world_sizes ===')
display(e2_combined[e3_cols])

## Visualization

In [ ]:
def plot_model(df, model_name, ax_est_act, ax_rank):
    for ws, gdf in df.groupby("world_size"):
        est = gdf["estimated_total_ms"].values
        act = gdf["actual_avg_ms"].values
        est_rank = rank_by(gdf["estimated_total_ms"]).values
        act_rank = rank_by(gdf["actual_avg_ms"]).values
        
        label = f"{model_name} ws={ws}"
        ax_est_act.scatter(est, act, label=label, s=80, alpha=0.7)
        ax_rank.scatter(est_rank, act_rank, label=label, s=80, alpha=0.7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_model(gpt2, "GPT-2", axes[0], axes[1])
plot_model(qwen, "Qwen2.5", axes[0], axes[1])

# Estimated vs Actual scatter
axes[0].plot([0, 3000], [0, 3000], 'k--', alpha=0.3, label='y=x')
axes[0].set_xlabel("Estimated step time (ms)")
axes[0].set_ylabel("Actual step time (ms)")
axes[0].set_title("E2: Estimated vs Actual Step Time")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rank scatter
axes[1].plot([0, 20], [0, 20], 'k--', alpha=0.3, label='y=x')
axes[1].set_xlabel("Estimated rank (1=best)")
axes[1].set_ylabel("Actual rank (1=best)")
axes[1].set_title("E2: Estimated Rank vs Actual Rank")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Per-world-size detailed tables

In [ ]:
def show_rank_table(df, model_name, world_size):
    gdf = df[(df["model"] == model_name) & (df["world_size"] == world_size)].copy()
    if len(gdf) == 0:
        print(f"No data for {model_name} ws={world_size}")
        return
    gdf["est_rank"] = rank_by(gdf["estimated_total_ms"])
    gdf["act_rank"] = rank_by(gdf["actual_avg_ms"])
    gdf["ratio"] = (gdf["actual_avg_ms"] / gdf["estimated_total_ms"]).round(2)
    cols = ["pp", "tp", "dp", "dp_outside", "estimated_total_ms", "actual_avg_ms", "ratio", "est_rank", "act_rank"]
    print(f"=== {model_name.upper()} | world_size={world_size} | {len(gdf)} candidates ===")
    display(gdf[cols].sort_values("est_rank"))

# Show all world sizes for GPT-2
for ws in sorted(gpt2["world_size"].unique()):
    show_rank_table(pd.concat([gpt2, qwen]), "gpt2", ws)

# Show all world sizes for Qwen
for ws in sorted(qwen["world_size"].unique()):
    show_rank_table(pd.concat([gpt2, qwen]), "qwen25", ws)

## Summary statistics across all groups

In [ ]:
print('=== E2 Summary ===')
for model in ['gpt2', 'qwen25']:
    sub = e2_all[e2_all['model'] == model]
    comb = e2_combined[e2_combined['model'] == model]
    print(f"{model.upper()} (per world_size):")
    print(f"  Spearman rho  mean={sub['spearman_rho'].mean():.3f}  min={sub['spearman_rho'].min():.3f}  max={sub['spearman_rho'].max():.3f}")
    print(f"  Kendall  tau  mean={sub['kendall_tau'].mean():.3f}  min={sub['kendall_tau'].min():.3f}  max={sub['kendall_tau'].max():.3f}")
    print(f"  Top-1 accuracy: {sub['top1_acc'].sum()}/{len(sub)} = {sub['top1_acc'].mean()*100:.0f}%")
    print(f"  Top-k accuracy: {sub['topk_acc'].mean()*100:.0f}%")
    print(f"  MAPE: {sub['mape_pct'].mean():.1f}% (mean)")
    if len(comb):
        c = comb.iloc[0]
        print(f"  COMBINED (all world_sizes): rho={c['spearman_rho']:.3f}  tau={c['kendall_tau']:.3f}  top1={c['top1_acc']}  mape={c['mape_pct']:.1f}%")
    print()

print('=== E3 Summary ===')
for model in ['gpt2', 'qwen25']:
    sub = e2_all[e2_all['model'] == model]
    comb = e2_combined[e2_combined['model'] == model]
    print(f"{model.upper()} (per world_size):")
    print(f"  Winner correct: {sub['winner_correct'].sum()}/{len(sub)}")
    regrets = sub[sub['regret_pct'].notna()]['regret_pct']
    print(f"  Mean regret: {regrets.mean():.1f}%")
    print(f"  Max regret:  {regrets.max():.1f}%")
    if len(comb):
        c = comb.iloc[0]
        print(f"  COMBINED (all world_sizes): winner_correct={c['winner_correct']}  regret={c['regret_pct']:.1f}%")
    print()